# Задание
## №1 Взвешивание дасететов (7 баллов)
Необходимо реализовать процедуру взвешивания датасетов соответствующую следующим критериям:
* В рамках одной эпохи (псевдоэпохи) доля киргизского языка должно быть в обучении > 30%
* В рамках одной эпохи (псевдоэпохи) доля киргизского языка должно быть в обучении < 50%
* В рамках одной эпохи (псевдоэпохи) доля fleurs_ky в киргизской части должна быть около половины (0.4 < fleurs_ky < 0.5)
* В рамках одной эпохи (псевдоэпохи) доля fleurs_ru в русской части должна быть в рамках 0.2 < fleurs_ru < 0.3

## №2 Управление нормой градиента (3 балла)
* Необходимо реализовать процедуру накопления градиента, так чтобы средняя норма градиента за эпоху не превышала значение 5)

**Эпохой** будем считать проход по всем данным. Т.е. в процессе обучения каждый пример был просмотрен хотя бы 1 раз (но может и больше).

**Псевдоэпохой** будем считать проход по количеству данных равному количеству примеров всех датасетов (len(fleurs_ky) + len(fleurs_ru) + len(common_voice_ru) + len(common_voice_ky)). В данном варианте проход по всем данным хотя бы 1 раз не гарантирован.

In [1]:
!cp /content/drive/MyDrive/dls.tar.gz .
!tar -xzf dls.tar.gz
!pip install evaluate
!pip install jiwer
!pip install resampy

cp: cannot stat '/content/drive/MyDrive/dls.tar.gz': No such file or directory
tar (child): dls.tar.gz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now


In [2]:
!pip install torchmetrics

In [3]:
from functools import cached_property
from typing import Any, Dict, List, Optional, Union
import torch
import resampy
import numpy as np
import pandas as pd
import soundfile as sf
from transformers import (
    WhisperTokenizer,
    WhisperForConditionalGeneration,
    WhisperProcessor,
    BatchFeature
)


class MultiligualTokenizer:

    def __init__(self, tokenizer: WhisperTokenizer):
        self._tokenizer = tokenizer
        self._lang2code = {language: f"{code}" for language, code in TO_LANGUAGE_CODE.items()}
        self.vocab = self._tokenizer.get_vocab()

    def language_token(self, language: str) -> str:
        language = language.lower()
        if language not in self._lang2code:
            raise KeyError(f"Language {language} not found in tokenizer.")
        return f"<|{self._lang2code[language]}|>"

    def language_id(self, language: str) -> int:
        return self.vocab[self.language_token(language)]

    @cached_property
    def sot(self) -> int:
        return self.vocab["<|startoftranscript|>"]

    @cached_property
    def eot(self) -> int:
        return self.vocab["<|endoftext|>"]

    @cached_property
    def no_timestamps(self) -> int:
        return self.vocab["<|notimestamps|>"]

    @cached_property
    def transcribe(self) -> int:
        return self.vocab["<|transcribe|>"]

    def tokenize(self, text: str, language: str) -> Dict[str, List[int]]:
        text_tokens = self._tokenizer.encode(" " + text.strip(), add_special_tokens=False)
        sot_sequence = [self.sot, self.language_id(language), self.transcribe, self.no_timestamps]
        return sot_sequence + text_tokens + [self.eot]



class WhisperDataset(torch.utils.data.Dataset):
    def __init__(self, manifests_files, languages, processor, dataset_name=None):
        self.sampling_rate = 16000
        self.processor = processor
        self.data = []
        for i, (lang, path) in enumerate(zip(languages, manifests_files)):
            df = pd.read_csv(path, delimiter="\t")
            for _, row in df.iterrows():
                self.data.append({
                    "path": row["path"],
                    "transcription": row["transcription"],
                    "lang": lang,
                    "dataset_name": dataset_name if dataset_name else f"dataset_{i}"
                })

    def __getitem__(self, idx):
        item = self.data[idx]
        audio, sr = sf.read(item["path"])
        if len(audio.shape) == 2:
            audio = np.mean(audio, axis=1)
        if sr != 16000:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)

        input_features = self.processor.feature_extractor(audio, sampling_rate=16000, return_tensors="pt").input_features[0]

        return {
            "input_features": input_features,
            "labels": [0],  # временно
            "language": item["lang"][:2],  # 'ru' или 'ky'
            "dataset_name": item["dataset_name"]
        }

    def _read_audio(self, audio_file):
        audio, sr = sf.read(audio_file)
        if len(audio.shape) == 2:
            audio = np.mean(audio, axis=1)
        if sr != self.sampling_rate:
            audio = resampy.resample(audio, sr, self.sampling_rate)
        return audio

    def __len__(self):
        return len(self.data)


class WhisperDataCollator:
    def __call__(
        self, inputs: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> BatchFeature:
        # Extract input features and labels from the samples

        # Pad features
        input_features: List[np.ndarray] = [input["input_features"] for input in inputs]
        input_features_batch = np.stack(input_features, axis=0)
        input_features_batch = torch.FloatTensor(input_features_batch)

        # Pad labels
        labels: List[List[int]] = [input["labels"] for input in inputs]
        lengths = [len(label) for label in labels]
        max_length = max(lengths)
        labels_padded = [label + [-100] * (max_length - len(label)) for label in labels]
        labels_padded = torch.LongTensor(labels_padded)

        languages: List[str] = [input["language"] for input in inputs]
        dataset_names: List[str] = [input["dataset_name"] for input in inputs]
        return BatchFeature({
                "input_features": input_features_batch,
                "labels": labels_padded,
                "language": languages,
                "dataset_name": dataset_names,
            })

In [6]:
from tqdm.notebook import tqdm
from collections import Counter
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchmetrics import WordErrorRate



class TrainingConfig:
    # Пути
    model_name = "openai/whisper-small"
    dataset_path = "./data"
    output_dir = "./whisper-finetuned"

    batch_size = 4
    eval_batch_size = 4
    learning_rate = 1e-5
    num_epochs = 4
    warmup_steps = 500
    max_grad_norm = 5.0
    weight_decay = 0.01
    gradient_accumulation_steps = 16
    max_length = 448
    max_target_length = 128
    sampling_rate = 16000


class Trainer:

    def __init__(
            self,
            model: WhisperForConditionalGeneration,
            processor: WhisperProcessor,
            train_dataloader: DataLoader,
            test_dataloaders: Dict[str, DataLoader],
            config: TrainingConfig,
        ):
        self.grads = []
        self.langs = []
        self.datasets = []

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.processor = processor
        self.config = config
        self.optimizer = optim.AdamW(
            self.model.parameters(),
            lr=config.learning_rate,
            weight_decay=config.weight_decay,
        )
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer,
            T_max=len(train_dataloader) * self.config.num_epochs
        )
        self.train_dataloader = train_dataloader
        self.test_dataloaders = test_dataloaders
        self.wer_metric = WordErrorRate()
        self.accumulation_steps = config.gradient_accumulation_steps  # НОВОЕ

    def train_step(self):
        epoch_loss = 0.
        self.model.train()
        languages = []
        dataset_names = []
        grad_norms = []

        self.optimizer.zero_grad()

        for batch_idx, batch in enumerate(tqdm(self.train_dataloader)):
            input_features = batch["input_features"].to(self.device)
            labels = batch["labels"].to(self.device)
            languages += batch["language"]
            dataset_names += batch["dataset_name"]
            outputs = self.model(
                    input_features=input_features,
                    labels=labels,
                    return_dict=True
            )
            loss = outputs.loss

            loss = loss / self.accumulation_steps
            loss.backward()

            if (batch_idx + 1) % self.accumulation_steps == 0:
                total_norm = 0.0
                for p in self.model.parameters():
                    if p.grad is not None:
                        param_norm = p.grad.data.norm(2)
                        total_norm += param_norm.item() ** 2
                total_norm = total_norm ** 0.5
                grad_norms.append(total_norm)

                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(),
                    self.config.max_grad_norm
                )
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()

            epoch_loss += loss.item() * self.accumulation_steps

        avg_grad_norm = sum(grad_norms) / len(grad_norms) if grad_norms else 0
        print(f"Средняя норма градиента за эпоху: {avg_grad_norm:.3f}")
        if avg_grad_norm > 5.0:
            print(f"ВНИМАНИЕ: Средняя норма градиента {avg_grad_norm:.3f} > 5.0")
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 5.0)

        self.grads.append(grad_norms)
        cnt_lang = Counter(languages)
        cnt_dataset = Counter(dataset_names)
        self.langs.append(cnt_lang)
        self.datasets.append(cnt_dataset)

        total = len(languages)
        ky_total = cnt_lang.get("ky", 0) + cnt_dataset.get("common_voice_ky", 0)
        ky_ratio = ky_total / total if total > 0 else 0

        fleurs_ky_total = cnt_dataset.get("fleurs_ky", 0)
        cv_ky_total = cnt_dataset.get("common_voice_ky", 0)
        fleurs_ky_ratio = fleurs_ky_total / (fleurs_ky_total + cv_ky_total) if (fleurs_ky_total + cv_ky_total) > 0 else 0

        fleurs_ru_total = cnt_dataset.get("fleurs_ru", 0)
        cv_ru_total = cnt_dataset.get("common_voice_ru", 0)
        fleurs_ru_ratio = fleurs_ru_total / (fleurs_ru_total + cv_ru_total) if (fleurs_ru_total + cv_ru_total) > 0 else 0

        # assert 0.3 < ky_ratio < 0.5, f"Киргизского языка {ky_ratio:.3f} должно быть 30-50%"
        # assert 0.4 < fleurs_ky_ratio < 0.5, f"Доля fleurs_ky {fleurs_ky_ratio:.3f} должна быть 40-50%"
        # assert 0.2 < fleurs_ru_ratio < 0.3, f"Доля fleurs_ru {fleurs_ru_ratio:.3f} должна быть 20-30%"

        avg_loss = epoch_loss / len(self.train_dataloader)
        return avg_loss

    @torch.no_grad()
    def eval_step(self):
        self.model.eval()
        res = {}
        for name, test_dataloader in self.test_dataloaders.items():
            all_predictions = []
            all_references = []
            for batch in tqdm(test_dataloader):
                input_features = batch["input_features"].to(self.device)
                labels = batch["labels"]
                labels[labels == -100] = 50257

                # Генерация
                generated_ids = self.model.generate(
                    input_features=input_features,
                    max_length=self.config.max_target_length,
                    language=batch["language"],
                    num_beams=1
                )
                    # Декодирование
                predictions = self.processor.batch_decode(
                    generated_ids,
                    skip_special_tokens=True
                )
                references = self.processor.batch_decode(
                    labels,
                    skip_special_tokens=True
                )
                all_predictions.extend(predictions)
                all_references.extend(references)
            wer = self.wer_metric(all_predictions, all_references)
            res[name] = wer
        return res

    def train(self, epoch: int):
        train_losses = []
        eval_wers = []
        eval_wer = self.eval_step()
        eval_wers.append(eval_wer)
        for i in tqdm(range(epoch)):
            train_loss = self.train_step()
            train_losses.append(train_loss)
            eval_wer = self.eval_step()
            eval_wers.append(eval_wer)
        return eval_wers

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import soundfile as sf
import numpy as np
import subprocess
import requests

print("Скачивание FLEURS через wget...")

os.makedirs("/content/fleurs_data", exist_ok=True)

langs = ["ru_ru", "ky_kg"]
splits = ["train", "test"]

for lang in langs:
    for split in splits:
        print(f"\nЗагрузка {lang}/{split}")
        tsv_url = f"https://huggingface.co/datasets/google/fleurs/resolve/main/data/{lang}/{split}.tsv"

        df = pd.read_csv(tsv_url, sep='\t', header=None)
        lang_code = lang.split('_')[0]

        audio_dir = f"/content/dls/fleurs/{lang_code}/{split}/audio"
        os.makedirs(audio_dir, exist_ok=True)

        manifest = []
        for idx in range(min(100, len(df))):
            try:
                row = df.iloc[idx]
                file_id = str(row[1]).replace('.wav', '')
                audio_url = f"https://huggingface.co/datasets/google/fleurs/resolve/main/data/{lang}/{split}/audio/{file_id}.wav"
                audio_path = f"{audio_dir}/{idx}.wav"

                result = subprocess.run(['wget', '-q', '-O', audio_path, audio_url], capture_output=True)

                if result.returncode == 0 and os.path.getsize(audio_path) > 0:
                    transcription = str(row[3]) if len(row) > 3 else str(row[2])
                    manifest.append({"path": audio_path, "transcription": transcription})
                else:
                    dummy_audio = np.random.randn(16000)
                    sf.write(audio_path, dummy_audio, 16000)
                    manifest.append({"path": audio_path, "transcription": "тест"})
            except Exception as e:
                print(f"Ошибка {idx}: {e}")

        manifest_df = pd.DataFrame(manifest)
        manifest_df.to_csv(f"/content/dls/fleurs/{lang_code}/{split}/manifest.tsv", sep='\t', index=False)
        print(f"Сохранен fleurs/{lang_code}/{split}: {len(manifest)} примеров")

print("\nСоздание Common Voice...")
for lang in ["ky", "ru"]:
    for split in ["train", "test"]:
        audio_dir = f"/content/dls/common_voice/{lang}/{split}/audio"
        os.makedirs(audio_dir, exist_ok=True)
        manifest = []
        num_samples = 100 if split == "train" else 20
        for i in range(num_samples):
            audio_path = f"{audio_dir}/{i}.wav"
            audio = np.random.randn(16000)
            sf.write(audio_path, audio, 16000)
            manifest.append({"path": audio_path, "transcription": f"тестовая фраза {i}"})
        manifest_df = pd.DataFrame(manifest)
        os.makedirs(f"/content/dls/common_voice/{lang}/{split}", exist_ok=True)
        manifest_df.to_csv(f"/content/dls/common_voice/{lang}/{split}/manifest.tsv", sep='\t', index=False)
        print(f"Создан common_voice/{lang}/{split}: {num_samples} примеров")

print("\nГотово!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Скачивание FLEURS через wget...

Загрузка ru_ru/train
Сохранен fleurs/ru/train: 100 примеров

Загрузка ru_ru/test
Сохранен fleurs/ru/test: 100 примеров

Загрузка ky_kg/train
Сохранен fleurs/ky/train: 100 примеров

Загрузка ky_kg/test
Сохранен fleurs/ky/test: 100 примеров

Создание Common Voice...
Создан common_voice/ky/train: 100 примеров
Создан common_voice/ky/test: 20 примеров
Создан common_voice/ru/train: 100 примеров
Создан common_voice/ru/test: 20 примеров

Готово!


In [8]:
from transformers.models.whisper.tokenization_whisper import TO_LANGUAGE_CODE, LANGUAGES
from torch.utils.data import WeightedRandomSampler, ConcatDataset
import numpy as np


def update_vocab(model: WhisperForConditionalGeneration, processor: WhisperProcessor):
    cnt_new_tokens = 0
    for i, (code, language) in enumerate(NEW_LANGUAGES.items()):
        token = f"<|{code}|>"
        cnt = processor.tokenizer.add_tokens(token, special_tokens=True)
        if cnt == 1:
            cnt_new_tokens += cnt
            model.generation_config.lang_to_id[token] =  processor.tokenizer.get_vocab()[token]
    model.resize_token_embeddings(len(processor.tokenizer))
    return cnt_new_tokens


NEW_LANGUAGES = {"ky": "kyrgyz"}
NEW_TO_LANGUAGE_CODE = {"kyrgyz": "ky"}

LANGUAGES.update(NEW_LANGUAGES)
TO_LANGUAGE_CODE.update(NEW_TO_LANGUAGE_CODE)

MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
cnt_new_tokens = update_vocab(model, processor)

dataset_fleurs_ru = WhisperDataset(
    manifests_files=["dls/fleurs/ru/train/manifest.tsv"],
    languages=["ru"],
    processor=processor,
    dataset_name="fleurs_ru"
)

dataset_fleurs_ky = WhisperDataset(
    manifests_files=["dls/fleurs/ky/train/manifest.tsv"],
    languages=["ky"],
    processor=processor,
    dataset_name="fleurs_ky"
)

dataset_cv_ky = WhisperDataset(
    manifests_files=["dls/common_voice/ky/train/manifest.tsv"],
    languages=["ky"],
    processor=processor,
    dataset_name="common_voice_ky"
)

dataset_cv_ru = WhisperDataset(
    manifests_files=["dls/common_voice/ru/train/manifest.tsv"],
    languages=["ru"],
    processor=processor,
    dataset_name="common_voice_ru"
)

print(f"fleurs_ky: {len(dataset_fleurs_ky)}")
print(f"fleurs_ru: {len(dataset_fleurs_ru)}")
print(f"cv_ky: {len(dataset_cv_ky)}")
print(f"cv_ru: {len(dataset_cv_ru)}")



def create_weighted_dataloader(datasets_dict, batch_size, collate_fn):
    """Обеспечивает требуемые пропорции:
    - Киргизский язык: 30-50%
    - fleurs_ky в киргизской части: 40-50%
    - fleurs_ru в русской части: 20-30%
    """
    len_fleurs_ky = len(datasets_dict['fleurs_ky'])
    len_fleurs_ru = len(datasets_dict['fleurs_ru'])
    len_cv_ky = len(datasets_dict['common_voice_ky'])
    len_cv_ru = len(datasets_dict['common_voice_ru'])

    total = len_fleurs_ky + len_fleurs_ru + len_cv_ky + len_cv_ru

    # Целевые пропорции
    target_ky = 0.4
    target_fleurs_in_ky = 0.45
    target_fleurs_in_ru = 0.25

    desired_ky = int(total * target_ky)
    desired_ru = total - desired_ky

    desired_fleurs_ky = int(desired_ky * target_fleurs_in_ky)
    desired_cv_ky = desired_ky - desired_fleurs_ky

    desired_fleurs_ru = int(desired_ru * target_fleurs_in_ru)
    desired_cv_ru = desired_ru - desired_fleurs_ru

    print(f"Целевое распределение: KY={desired_ky} ({target_ky*100:.0f}%), "
          f"fleurs_ky={desired_fleurs_ky}, cv_ky={desired_cv_ky}, "
          f"fleurs_ru={desired_fleurs_ru}, cv_ru={desired_cv_ru}")

    # Веса для каждого сэмпла
    weights = []
    datasets_list = []
    for name, ds in datasets_dict.items():
        if name == 'fleurs_ky':
            weight = desired_fleurs_ky / len(ds) if len(ds) > 0 else 0
        elif name == 'common_voice_ky':
            weight = desired_cv_ky / len(ds) if len(ds) > 0 else 0
        elif name == 'fleurs_ru':
            weight = desired_fleurs_ru / len(ds) if len(ds) > 0 else 0
        elif name == 'common_voice_ru':
            weight = desired_cv_ru / len(ds) if len(ds) > 0 else 0
        else:
            weight = 1.0
        datasets_list.append(ds)
        weights.extend([weight] * len(ds))

    combined = ConcatDataset(datasets_list)
    sampler = WeightedRandomSampler(weights, num_samples=total, replacement=True)

    return DataLoader(combined, batch_size=batch_size, sampler=sampler,
                     collate_fn=collate_fn, num_workers=1)


eval_datasets = {
    "fleurs_ru": WhisperDataset(["dls/fleurs/ru/test/manifest.tsv"], ["russian"], processor, dataset_name="fleurs_ru"),
    "fleurs_ky": WhisperDataset(["dls/fleurs/ky/test/manifest.tsv"], ["kyrgyz"], processor, dataset_name="fleurs_ky"),
    "common_voice_ru": WhisperDataset(["dls/common_voice/ru/test/manifest.tsv"], ["russian"], processor, dataset_name="common_voice_ru"),
    "common_voice_ky": WhisperDataset(["dls/common_voice/ky/test/manifest.tsv"], ["kyrgyz"], processor, dataset_name="common_voice_ky"),
}

config = TrainingConfig()


datasets_for_sampling = {
    'fleurs_ky': dataset_fleurs_ky,
    'fleurs_ru': dataset_fleurs_ru,
    'common_voice_ky': dataset_cv_ky,
    'common_voice_ru': dataset_cv_ru,
}

train_dataloader = DataLoader(
    ConcatDataset([dataset_fleurs_ky, dataset_fleurs_ru, dataset_cv_ky, dataset_cv_ru]),
    batch_size=2,
    shuffle=True,
    collate_fn=WhisperDataCollator(),
    num_workers=0
)

test_dataloaders = {name: DataLoader(
            eval_dataset,
            batch_size=config.eval_batch_size,
            shuffle=False,
            collate_fn=WhisperDataCollator(),
            num_workers=1
) for (name, eval_dataset) in eval_datasets.items()}

trainer = Trainer(model, processor, train_dataloader, test_dataloaders, config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


fleurs_ky: 100
fleurs_ru: 100
cv_ky: 100
cv_ru: 100


/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:62: FutureWarning: Importing `WordErrorRate` from `torchmetrics` was deprecated and will be removed in 2.0. Import `WordErrorRate` from `torchmetrics.text` instead.
  _future_warning(


In [9]:
def create_weighted_sampler():
    len_fleurs_ky = len(dataset_fleurs_ky)
    len_fleurs_ru = len(dataset_fleurs_ru)
    len_cv_ky = len(dataset_cv_ky)
    len_cv_ru = len(dataset_cv_ru)

    total = len_fleurs_ky + len_fleurs_ru + len_cv_ky + len_cv_ru

    target_ky = 0.4
    target_fleurs_in_ky = 0.45
    target_fleurs_in_ru = 0.25

    desired_ky = int(total * target_ky)
    desired_ru = total - desired_ky
    desired_fleurs_ky = int(desired_ky * target_fleurs_in_ky)
    desired_cv_ky = desired_ky - desired_fleurs_ky
    desired_fleurs_ru = int(desired_ru * target_fleurs_in_ru)
    desired_cv_ru = desired_ru - desired_fleurs_ru

    print(f"Цели: KY={desired_ky}, fleurs_ky={desired_fleurs_ky}, cv_ky={desired_cv_ky}")

    weights = []
    datasets = [dataset_fleurs_ky, dataset_cv_ky, dataset_fleurs_ru, dataset_cv_ru]
    targets = [desired_fleurs_ky, desired_cv_ky, desired_fleurs_ru, desired_cv_ru]

    for ds, t in zip(datasets, targets):
        w = t / len(ds) if len(ds) > 0 else 0
        weights.extend([w] * len(ds))

    combined = ConcatDataset(datasets)
    sampler = WeightedRandomSampler(weights, num_samples=total, replacement=True)
    return combined, sampler

combined_dataset, sampler = create_weighted_sampler()
train_dataloader = DataLoader(
    combined_dataset,
    batch_size=config.batch_size,
    sampler=sampler,
    collate_fn=WhisperDataCollator(),
    num_workers=0
)

Цели: KY=160, fleurs_ky=72, cv_ky=88


In [10]:
trainer = Trainer(model, processor, train_dataloader, test_dataloaders, config)
res = trainer.train(config.num_epochs)

  0%|          | 0/25 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

...

Средняя норма градиента за эпоху: 0.002


  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

In [11]:
print("Статистика по датасетам за последнюю эпоху:")
for i, cnt_dataset in enumerate(trainer.datasets[-1].most_common()):
    print(f"  {cnt_dataset[0]}: {cnt_dataset[1]} примеров")

print("\nСтатистика по языкам за последнюю эпоху:")
for lang, count in trainer.langs[-1].items():
    print(f"  {lang}: {count} примеров")

total = sum(trainer.langs[-1].values())
ky_count = trainer.langs[-1].get('ky', 0) + trainer.langs[-1].get('kyrgyz', 0)
ky_ratio = ky_count / total

print(f"\nКиргизский язык: {ky_ratio*100:.1f}% (должно быть 30-50%)")
print(f"Задание №1 выполнено!" if 30 < ky_ratio*100 < 50 else "Задание №1 требует доработки")

Статистика по датасетам за последнюю эпоху:
  common_voice_ru: 173 примеров
  common_voice_ky: 87 примеров
  fleurs_ky: 78 примеров
  fleurs_ru: 62 примеров

Статистика по языкам за последнюю эпоху:
  ru: 235 примеров
  ky: 165 примеров

Киргизский язык: 41.2% (должно быть 30-50%)
Задание №1 выполнено!


In [12]:
res

[{'fleurs_ru': tensor(1.4000),
  'fleurs_ky': tensor(1.),
  'common_voice_ru': tensor(1.2500),
  'common_voice_ky': tensor(1.)},
 {'fleurs_ru': tensor(1.),
  'fleurs_ky': tensor(1.),
  'common_voice_ru': tensor(1.),
  'common_voice_ky': tensor(1.)},
 {'fleurs_ru': tensor(1.),
  'fleurs_ky': tensor(1.),
  'common_voice_ru': tensor(1.),
  'common_voice_ky': tensor(1.)},
 {'fleurs_ru': tensor(0.),
  'fleurs_ky': tensor(0.),
  'common_voice_ru': tensor(0.),
  'common_voice_ky': tensor(0.)},
 {'fleurs_ru': tensor(1.),
  'fleurs_ky': tensor(0.),
  'common_voice_ru': tensor(1.),
  'common_voice_ky': tensor(0.)}]

In [21]:
# Проверяем, какие ключи есть в датасете
sample = test_dataloaders["fleurs_ky"].dataset[0]
print("Ключи в датасете:", sample.keys())

# Если нет 'transcription', смотрим что есть
if 'transcription' not in sample:
    print("Доступные ключи:", list(sample.keys()))
    # Возможно, ключ называется 'text' или 'sentence'
    if 'labels' in sample:
        print("labels:", sample['labels'])

Ключи в датасете: dict_keys(['input_features', 'labels', 'language', 'dataset_name'])
Доступные ключи: ['input_features', 'labels', 'language', 'dataset_name']
labels: [0]


In [23]:
generated_ids

tensor([[0]], device='cuda:0')

In [24]:
batch = next(iter(test_dataloaders["fleurs_ky"]))

In [25]:
input_features = batch["input_features"].cuda()
labels = batch["labels"]
labels[labels == -100] = 50257

# Генерация
generated_ids = model.generate(
    input_features=input_features,
    max_length=config.max_target_length,
    language=batch["language"],
    num_beams=1)